# Bond–Equity Volatility Divergence

**Research question:** When Treasury-market volatility becomes unusually high relative to equity volatility, does subsequent SPY risk increase?

This notebook preserves the full research trail for POC01: data QA, point-in-time feature construction, the original 1–5D test, a pre-specified 30D extension, development/confirmation analysis, component attribution, and overlapping-label robustness.

**Main result:** the proposed divergence feature, `VolDiv_prior = MOVE_Z60_prior - VIX_Z60_prior`, was not supported as a stable forward SPY-risk predictor. MOVE and VIX individually retained positive relationships with subsequent realized volatility, while the subtraction-based feature largely removed that information.

> This is a research/education project, not investment advice.


## POC01-A — Data, QA, and Original 1–5D Study


### Reproducibility note
Market data are downloaded at runtime with `yfinance`. Results can vary slightly if the upstream provider revises historical data. The analysis uses an inner join across MOVE, VIX, and SPY and does not forward-fill missing market observations.


In [2]:
!pip install yfinance -q

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf

print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)

pandas: 2.2.3
yfinance: 0.2.66


In [4]:
start = "2004-01-01"
end = "2026-09-28"

spy = yf.download("SPY", start=start, end=end, auto_adjust=True)["Close"]
vix = yf.download("^VIX", start=start, end=end, auto_adjust=False)["Close"]
move = yf.download("^MOVE", start=start, end=end, auto_adjust=False)["Close"]

print("SPY:", spy.shape, spy.index.min(), spy.index.max())
print("VIX:", vix.shape, vix.index.min(), vix.index.max())
print("MOVE:", move.shape, move.index.min(), move.index.max())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

SPY: (5719, 1) 2004-01-02 00:00:00 2026-09-25 00:00:00
VIX: (5721, 1) 2004-01-02 00:00:00 2026-09-25 00:00:00
MOVE: (5626, 1) 2004-01-02 00:00:00 2026-09-25 00:00:00


In [5]:
# Step 1：先把“不一致在哪里”搞清楚
# Convert to Series in case yfinance returned 1-column DataFrames
spy = spy.squeeze()
vix = vix.squeeze()
move = move.squeeze()

print("SPY not in MOVE:", len(spy.index.difference(move.index)))
print("MOVE not in SPY:", len(move.index.difference(spy.index)))

print("\nFirst 20 SPY dates missing from MOVE:")
print(spy.index.difference(move.index)[:20])

print("\nVIX not in SPY:")
print(vix.index.difference(spy.index))

SPY not in MOVE: 93
MOVE not in SPY: 0

First 20 SPY dates missing from MOVE:
DatetimeIndex(['2004-10-11', '2004-11-11', '2005-10-10', '2005-11-11',
               '2006-10-09', '2007-10-08', '2007-11-12', '2008-10-13',
               '2009-10-12', '2009-11-11', '2010-10-11', '2010-11-11',
               '2011-10-10', '2011-11-11', '2012-10-08', '2012-11-12',
               '2012-11-23', '2012-11-26', '2012-12-07', '2012-12-13'],
              dtype='datetime64[ns]', name='Date', freq=None)

VIX not in SPY:
DatetimeIndex(['2026-05-25', '2026-09-07'], dtype='datetime64[ns]', name='Date', freq=None)


In [6]:
# Step 2: inter join 3 tables
df = pd.concat(
    [move, vix, spy],
    axis=1,
    join="inner"
)

df.columns = ["MOVE", "VIX", "SPY"]

print("Final dataset:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print("\nMissing:")
print(df.isna().sum())

Final dataset: (5626, 3)
Date range: 2004-01-02 00:00:00 to 2026-09-25 00:00:00

Missing:
MOVE    0
VIX     0
SPY     0
dtype: int64


In [7]:
# Step 3: create features
window = 60

df["MOVE_mean60"] = df["MOVE"].rolling(window).mean()
df["MOVE_std60"]  = df["MOVE"].rolling(window).std()

df["VIX_mean60"] = df["VIX"].rolling(window).mean()
df["VIX_std60"]  = df["VIX"].rolling(window).std()

df["MOVE_Z60"] = (
    (df["MOVE"] - df["MOVE_mean60"])
    / df["MOVE_std60"]
)

df["VIX_Z60"] = (
    (df["VIX"] - df["VIX_mean60"])
    / df["VIX_std60"]
)

df["VolDivergence"] = df["MOVE_Z60"] - df["VIX_Z60"]

In [8]:
# Step 4: sanity checks
print(df[
    ["MOVE", "VIX", "MOVE_Z60", "VIX_Z60", "VolDivergence"]
].tail(10))

print("\nMissing after feature engineering:")
print(df[
    ["MOVE_Z60", "VIX_Z60", "VolDivergence"]
].isna().sum())

                  MOVE        VIX  MOVE_Z60   VIX_Z60  VolDivergence
Date                                                                
2026-09-14   83.900002  17.100000  2.291647  0.478224       1.813423
2026-09-15   83.709999  17.200001  2.152768  0.535614       1.617153
2026-09-16   80.730003  17.709999  1.451039  0.872313       0.578726
2026-09-17   76.220001  15.440000  0.465781 -0.632634       1.098416
2026-09-18   80.639999  14.810000  1.367384 -1.040929       2.408313
2026-09-21   81.199997  14.870000  1.440362 -0.972268       2.412630
2026-09-22   78.559998  14.210000  0.846972 -1.405007       2.251979
2026-09-23   95.449997  15.180000  3.884402 -0.675684       4.560085
2026-09-24  104.580002  15.670000  4.482355 -0.308306       4.790661
2026-09-25   96.000000  14.870000  2.882078 -0.866711       3.748789

Missing after feature engineering:
MOVE_Z60         59
VIX_Z60          59
VolDivergence    59
dtype: int64


In [9]:
valid = df["VolDivergence"].dropna()

df["VolDiv_Percentile"] = (
    df["VolDivergence"].rank(pct=True) * 100
)

print(
    df.loc[
        "2026-09-24",
        [
            "MOVE",
            "VIX",
            "MOVE_Z60",
            "VIX_Z60",
            "VolDivergence",
            "VolDiv_Percentile"
        ]
    ]
)

MOVE                 104.580002
VIX                   15.670000
MOVE_Z60               4.482355
VIX_Z60               -0.308306
VolDivergence          4.790661
VolDiv_Percentile     99.982037
Name: 2026-09-24 00:00:00, dtype: float64


In [10]:
# Step 5: how divergent Sep 24 2026 was compare to historical 5600 records
top20 = df.nlargest(
    20,
    "VolDivergence"
)[
    [
        "MOVE",
        "VIX",
        "MOVE_Z60",
        "VIX_Z60",
        "VolDivergence",
        "VolDiv_Percentile"
    ]
]

print(top20)

                  MOVE        VIX  MOVE_Z60   VIX_Z60  VolDivergence  \
Date                                                                   
2009-06-01  190.300003  30.040001  3.626553 -1.437053       5.063606   
2026-09-24  104.580002  15.670000  4.482355 -0.308306       4.790661   
2026-09-23   95.449997  15.180000  3.884402 -0.675684       4.560085   
2009-06-02  183.899994  29.629999  3.038778 -1.491296       4.530074   
2021-02-16   56.779999  21.459999  3.816102 -0.577627       4.393729   
2010-12-15  125.199997  17.940001  2.962594 -1.254258       4.216852   
2010-12-17  119.099998  16.110001  2.093488 -2.054025       4.147513   
2009-06-05  183.399994  29.620001  2.707191 -1.404633       4.111823   
2009-05-29  165.399994  28.920000  2.372454 -1.666574       4.039029   
2010-12-16  122.699997  17.389999  2.547012 -1.488775       4.035787   
2016-11-14   89.220001  14.480000  4.067207  0.036322       4.030886   
2010-12-14  118.400002  17.610001  2.510179 -1.477516       3.98

In [11]:
# Step 6: 今天相对于“包含今天在内的最近60日环境”有多异常？/今天突然发生的东西，相对于昨天为止的市场状态，到底有多离谱？
df["MOVE_mean60_prior"] = (
    df["MOVE"].shift(1).rolling(60).mean()
)

df["MOVE_std60_prior"] = (
    df["MOVE"].shift(1).rolling(60).std()
)

df["VIX_mean60_prior"] = (
    df["VIX"].shift(1).rolling(60).mean()
)

df["VIX_std60_prior"] = (
    df["VIX"].shift(1).rolling(60).std()
)

df["MOVE_Z60_prior"] = (
    (df["MOVE"] - df["MOVE_mean60_prior"])
    / df["MOVE_std60_prior"]
)

df["VIX_Z60_prior"] = (
    (df["VIX"] - df["VIX_mean60_prior"])
    / df["VIX_std60_prior"]
)

df["VolDiv_prior"] = (
    df["MOVE_Z60_prior"] - df["VIX_Z60_prior"]
)

In [12]:
print(
    df.loc[
        "2026-09-23":"2026-09-25",
        [
            "MOVE_Z60",
            "MOVE_Z60_prior",
            "VIX_Z60",
            "VIX_Z60_prior",
            "VolDivergence",
            "VolDiv_prior"
        ]
    ]
)

            MOVE_Z60  MOVE_Z60_prior   VIX_Z60  VIX_Z60_prior  VolDivergence  \
Date                                                                           
2026-09-23  3.884402        4.535032 -0.675684      -0.701233       4.560085   
2026-09-24  4.482355        5.630251 -0.308306      -0.317883       4.790661   
2026-09-25  2.882078        3.151607 -0.866711      -0.892311       3.748789   

            VolDiv_prior  
Date                      
2026-09-23      5.236265  
2026-09-24      5.948134  
2026-09-25      4.043918  


# 正式冻结 POC01 Feature = VolDiv_prior = Z_prior60(MOVE) − Z_prior60(VIX)

# **POC01-A — Original 1/3/5D Study**

In [13]:
# Step 1: 算 SPY daily return
df["SPY_Return_1D"] = df["SPY"].pct_change()

In [14]:
# Step 2: future 1D, 3D, 5D return forecast
df["Fwd_Return_1D"] = (
    df["SPY"].shift(-1) / df["SPY"] - 1
)

df["Fwd_Return_3D"] = (
    df["SPY"].shift(-3) / df["SPY"] - 1
)

df["Fwd_Return_5D"] = (
    df["SPY"].shift(-5) / df["SPY"] - 1
)

In [15]:
# Step 3: sanity check
# Frozen POC01 feature: prior 60-day z-score

window = 60

move_prior = df["MOVE"].shift(1).rolling(window)
vix_prior  = df["VIX"].shift(1).rolling(window)

df["MOVE_Z60_prior"] = (
    (df["MOVE"] - move_prior.mean())
    / move_prior.std()
)

df["VIX_Z60_prior"] = (
    (df["VIX"] - vix_prior.mean())
    / vix_prior.std()
)

df["VolDiv_prior"] = (
    df["MOVE_Z60_prior"] - df["VIX_Z60_prior"]
)



In [16]:
print(
    df.loc[
        "2026-09-23":"2026-09-25",
        [
            "SPY",
            "VolDiv_prior",
            "Fwd_Return_1D",
            "Fwd_Return_3D",
            "Fwd_Return_5D"
        ]
    ]
)

                   SPY  VolDiv_prior  Fwd_Return_1D  Fwd_Return_3D  \
Date                                                                 
2026-09-23  767.809998      5.236265      -0.000821            NaN   
2026-09-24  767.179993      5.948134       0.005435            NaN   
2026-09-25  771.349976      4.043918            NaN            NaN   

            Fwd_Return_5D  
Date                       
2026-09-23            NaN  
2026-09-24            NaN  
2026-09-25            NaN  


In [17]:
#Step 4: Volatility divergence quintiles vs forward SPY returns
analysis = df.dropna(
    subset=[
        "VolDiv_prior",
        "Fwd_Return_1D",
        "Fwd_Return_3D",
        "Fwd_Return_5D"
    ]
).copy()

analysis["VolDiv_Q"] = pd.qcut(
    analysis["VolDiv_prior"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

quintile_returns = (
    analysis
    .groupby("VolDiv_Q", observed=True)
    [["Fwd_Return_1D", "Fwd_Return_3D", "Fwd_Return_5D"]]
    .mean()
)

print((quintile_returns * 100).round(3))

          Fwd_Return_1D  Fwd_Return_3D  Fwd_Return_5D
VolDiv_Q                                             
Q1                0.121          0.255          0.408
Q2                0.025          0.134          0.201
Q3                0.011          0.065          0.187
Q4                0.026          0.120          0.173
Q5                0.063          0.145          0.220


In [18]:
# Step 5: Forward Min Return
future_returns = pd.concat(
    [
        df["SPY"].shift(-k) / df["SPY"] - 1
        for k in range(1, 6)
    ],
    axis=1
)

df["Forward_Min_Return_5D"] = future_returns.min(
    axis=1,
    skipna=False
)




print(
    df[
        ["SPY", "Forward_Min_Return_5D"]
    ].tail(7)
)


analysis = df.dropna(
    subset=[
        "VolDiv_prior",
        "Forward_Min_Return_5D"
    ]
).copy()

analysis["VolDiv_Q"] = pd.qcut(
    analysis["VolDiv_prior"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

min_return_by_q = (
    analysis
    .groupby("VolDiv_Q", observed=True)
    ["Forward_Min_Return_5D"]
    .mean()
)

print((min_return_by_q * 100).round(3))

                   SPY  Forward_Min_Return_5D
Date                                         
2026-09-17  760.710999               0.001287
2026-09-18  761.690002               0.007208
2026-09-21  773.500000                    NaN
2026-09-22  773.380005                    NaN
2026-09-23  767.809998                    NaN
2026-09-24  767.179993                    NaN
2026-09-25  771.349976                    NaN
VolDiv_Q
Q1   -0.906
Q2   -1.068
Q3   -1.025
Q4   -0.928
Q5   -0.795
Name: Forward_Min_Return_5D, dtype: float64


In [19]:
# Step 6: Lable 3 Forward Realized Volatility
# a. future 5 daily returns

future_daily_returns = pd.concat(
    [
        df["SPY_Return_1D"].shift(-k)
        for k in range(1, 6)
    ],
    axis=1
)

# b. future 5-day realized volatility

df["Forward_RV_5D"] = future_daily_returns.std(
    axis=1,
    skipna=False
)

# c. sanity check

print(
    df[
        ["SPY", "Forward_RV_5D"]
    ].tail(7)
)

                   SPY  Forward_RV_5D
Date                                 
2026-09-17  760.710999       0.008363
2026-09-18  761.690002       0.008514
2026-09-21  773.500000            NaN
2026-09-22  773.380005            NaN
2026-09-23  767.809998            NaN
2026-09-24  767.179993            NaN
2026-09-25  771.349976            NaN


In [20]:
# d. quintile
analysis = df.dropna(
    subset=[
        "VolDiv_prior",
        "Forward_RV_5D"
    ]
).copy()

analysis["VolDiv_Q"] = pd.qcut(
    analysis["VolDiv_prior"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

rv_by_q = (
    analysis
    .groupby("VolDiv_Q", observed=True)
    ["Forward_RV_5D"]
    .mean()
)

print((rv_by_q * 100).round(3))

VolDiv_Q
Q1    0.986
Q2    1.038
Q3    0.930
Q4    0.877
Q5    0.786
Name: Forward_RV_5D, dtype: float64


到目前为止，POC01 给我们的答案
当 Treasury volatility 相对于 equity volatility 异常高时，未来1–5日 SPY risk 是否增加？

目前三个预先定义的 labels：

1.   **Forward Return：**没有出现预期的持续恶化。5D 从 Q1 的 +0.408% 到 Q5 的 +0.220%，但不是单调关系。
2. **Forward Min Return：**没有恶化，甚至 Q5 更温和。1 -0.906% → Q5 -0.793%
3. **Forward Realized Volatility：**不仅没增加，反而 Q5 更低。Q1 0.986% → Q5 0.786%

所以按照目前 exploratory evidence：
我们的原始“Bond Volatility Contagion → Equity Risk”假设没有得到支持。

而且第三个结果甚至提示了一个反方向关系。

In [21]:
diagnostic = (
    analysis
    .groupby("VolDiv_Q", observed=True)
    [
        [
            "MOVE_Z60_prior",
            "VIX_Z60_prior",
            "Forward_RV_5D"
        ]
    ]
    .mean()
)

print(diagnostic.round(3))

          MOVE_Z60_prior  VIX_Z60_prior  Forward_RV_5D
VolDiv_Q                                              
Q1                -0.808          1.142          0.010
Q2                -0.640          0.035          0.010
Q3                -0.343         -0.294          0.009
Q4                 0.213         -0.420          0.009
Q5                 1.161         -0.675          0.008


In [22]:
# 单独看今天的 MOVE Z 和 VIX Z，哪个跟未来5日股票波动关系更强？
print(
    analysis[
        [
            "MOVE_Z60_prior",
            "VIX_Z60_prior",
            "VolDiv_prior",
            "Forward_RV_5D"
        ]
    ].corr()["Forward_RV_5D"].round(3)
)

MOVE_Z60_prior    0.272
VIX_Z60_prior     0.369
VolDiv_prior     -0.103
Forward_RV_5D     1.000
Name: Forward_RV_5D, dtype: float64


In [25]:
print(df.shape)

required = [
    "MOVE",
    "VIX",
    "SPY",
    "SPY_Return_1D",
    "MOVE_Z60_prior",
    "VIX_Z60_prior",
    "VolDiv_prior"
]

print(df[required].tail())

(5626, 27)
                  MOVE    VIX         SPY  SPY_Return_1D  MOVE_Z60_prior  \
Date                                                                       
2026-09-21   81.199997  14.87  773.500000       0.015505        1.487217   
2026-09-22   78.559998  14.21  773.380005      -0.000155        0.874084   
2026-09-23   95.449997  15.18  767.809998      -0.007202        4.535032   
2026-09-24  104.580002  15.67  767.179993      -0.000821        5.630251   
2026-09-25   96.000000  14.87  771.349976       0.005435        3.151607   

            VIX_Z60_prior  VolDiv_prior  
Date                                     
2026-09-21      -0.998543      2.485760  
2026-09-22      -1.448834      2.322918  
2026-09-23      -0.701233      5.236265  
2026-09-24      -0.317883      5.948134  
2026-09-25      -0.892311      4.043918  


# **POC01-B 30D Horizon Extension**

# **Prediction**
1. 30D Return：↓ —— VolDiv越高，未来30D平均return倾向较低
2. 30D Min Return：不确定。
3. 30D RV：不确定。  
4. Feature 不变：VolDiv_prior = MOVE_Z60_prior - VIX_Z60_prior
5. Primary horizon = 30 trading days。
6. 60D暂不测试，保留作 robustness check。
7. 在看到30D结果之前不修改feature/window/label定义。

In [26]:
# ==========================================
# POC01-B: 30D Forward Labels
# ==========================================

# 1. Forward 30D Return
df["Fwd_Return_30D"] = (
    df["SPY"].shift(-30) / df["SPY"] - 1
)

# 2. Forward 30D Minimum Return
future_returns_30 = pd.concat(
    [
        df["SPY"].shift(-k) / df["SPY"] - 1
        for k in range(1, 31)
    ],
    axis=1
)

df["Forward_Min_Return_30D"] = (
    future_returns_30.min(axis=1, skipna=False)
)

# 3. Forward 30D Realized Volatility
future_daily_returns_30 = pd.concat(
    [
        df["SPY_Return_1D"].shift(-k)
        for k in range(1, 31)
    ],
    axis=1
)

df["Forward_RV_30D"] = (
    future_daily_returns_30.std(axis=1, skipna=False)
)

In [27]:
# QA checking if days match
cols_30d = [
    "SPY",
    "VolDiv_prior",
    "Fwd_Return_30D",
    "Forward_Min_Return_30D",
    "Forward_RV_30D"
]

print(df[cols_30d].tail(35))

print("\nNaN counts:")
print(
    df[
        [
            "Fwd_Return_30D",
            "Forward_Min_Return_30D",
            "Forward_RV_30D"
        ]
    ].isna().sum()
)

                   SPY  VolDiv_prior  Fwd_Return_30D  Forward_Min_Return_30D  \
Date                                                                           
2026-08-07  771.344604      1.332291        0.002794               -0.024843   
2026-08-10  771.115173      1.611405        0.002937               -0.024553   
2026-08-11  768.651306      2.171095       -0.001095               -0.021426   
2026-08-12  770.576477      1.374023       -0.004408               -0.023871   
2026-08-13  775.953186      0.691871       -0.005932               -0.030635   
2026-08-14  774.416992      0.948221             NaN                     NaN   
2026-08-17  770.756042      1.740652             NaN                     NaN   
2026-08-18  765.549011      1.221616             NaN                     NaN   
2026-08-19  767.154968      0.881146             NaN                     NaN   
2026-08-20  760.710999      0.711544             NaN                     NaN   
2026-08-21  763.823242      1.265207    

In [28]:
# ==========================================
# POC01-B: Development / Confirmation Split
# ==========================================

label_cols = [
    "Fwd_Return_30D",
    "Forward_Min_Return_30D",
    "Forward_RV_30D"
]

analysis_30 = df.dropna(
    subset=["VolDiv_prior"] + label_cols
).copy()

dev = analysis_30.loc[
    analysis_30.index < "2023-01-01"
].copy()

confirm = analysis_30.loc[
    (analysis_30.index >= "2023-01-01") &
    (analysis_30.index < "2026-01-01")
].copy()

live_2026 = analysis_30.loc[
    analysis_30.index >= "2026-01-01"
].copy()

print("Development:", len(dev),
      dev.index.min(), "to", dev.index.max())

print("Confirmation:", len(confirm),
      confirm.index.min(), "to", confirm.index.max())

print("Live 2026:", len(live_2026),
      live_2026.index.min(), "to", live_2026.index.max())

Development: 4635 2004-03-30 00:00:00 to 2022-12-30 00:00:00
Confirmation: 747 2023-01-03 00:00:00 to 2025-12-31 00:00:00
Live 2026: 154 2026-01-02 00:00:00 to 2026-08-13 00:00:00


In [29]:
# Quintile boundaries learned ONLY from development data

_, q_edges = pd.qcut(
    dev["VolDiv_prior"],
    q=5,
    retbins=True
)

print("Development quintile boundaries:")
print(q_edges)

Development quintile boundaries:
[-14.46975773  -1.05385227  -0.34381457   0.30086285   1.11790652
   5.44258392]


In [30]:
# 因为未来可能出现比2004–22历史最大值更极端的 VolDiv，所以我们把最外侧边界扩展到 ±∞
q_edges_fixed = q_edges.copy()

q_edges_fixed[0] = -np.inf
q_edges_fixed[-1] = np.inf

labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]

dev["VolDiv_Q"] = pd.cut(
    dev["VolDiv_prior"],
    bins=q_edges_fixed,
    labels=labels,
    include_lowest=True
)

confirm["VolDiv_Q"] = pd.cut(
    confirm["VolDiv_prior"],
    bins=q_edges_fixed,
    labels=labels,
    include_lowest=True
)

In [31]:
print("\nDEV counts:")
print(dev["VolDiv_Q"].value_counts().sort_index())

print("\nCONFIRM counts:")
print(confirm["VolDiv_Q"].value_counts().sort_index())

print("\nFixed edges:")
print(q_edges_fixed)


DEV counts:
VolDiv_Q
Q1    927
Q2    927
Q3    927
Q4    927
Q5    927
Name: count, dtype: int64

CONFIRM counts:
VolDiv_Q
Q1    167
Q2    165
Q3    191
Q4    150
Q5     74
Name: count, dtype: int64

Fixed edges:
[       -inf -1.05385227 -0.34381457  0.30086285  1.11790652         inf]


In [32]:
# ==========================================
# POC01-B: 30D Results
# ==========================================

result_cols = [
    "Fwd_Return_30D",
    "Forward_Min_Return_30D",
    "Forward_RV_30D"
]

dev_results = (
    dev.groupby("VolDiv_Q", observed=True)[result_cols]
       .mean() * 100
)

confirm_results = (
    confirm.groupby("VolDiv_Q", observed=True)[result_cols]
           .mean() * 100
)

print("===== DEVELOPMENT: 2004–2022 =====")
print(dev_results.round(3))

print("\n===== CONFIRMATION: 2023–2025 =====")
print(confirm_results.round(3))

===== DEVELOPMENT: 2004–2022 =====
          Fwd_Return_30D  Forward_Min_Return_30D  Forward_RV_30D
VolDiv_Q                                                        
Q1                 2.015                  -2.885           1.004
Q2                 1.149                  -3.539           1.086
Q3                 1.090                  -3.549           1.052
Q4                 0.893                  -3.667           1.035
Q5                 0.950                  -3.012           0.899

===== CONFIRMATION: 2023–2025 =====
          Fwd_Return_30D  Forward_Min_Return_30D  Forward_RV_30D
VolDiv_Q                                                        
Q1                 1.565                  -2.931           0.879
Q2                 1.397                  -3.088           0.929
Q3                 2.941                  -2.196           0.863
Q4                 3.482                  -1.826           0.816
Q5                 3.555                  -1.496           0.789


In [33]:
# ==========================================
# POC01-B: 30D Feature Attribution
# ==========================================

features = [
    "MOVE_Z60_prior",
    "VIX_Z60_prior",
    "VolDiv_prior"
]

print("===== DEVELOPMENT =====")
print(
    dev[features + ["Forward_RV_30D"]]
    .corr()["Forward_RV_30D"]
    .loc[features]
    .round(3)
)

print("\n===== CONFIRMATION =====")
print(
    confirm[features + ["Forward_RV_30D"]]
    .corr()["Forward_RV_30D"]
    .loc[features]
    .round(3)
)

===== DEVELOPMENT =====
MOVE_Z60_prior    0.251
VIX_Z60_prior     0.290
VolDiv_prior     -0.043
Name: Forward_RV_30D, dtype: float64

===== CONFIRMATION =====
MOVE_Z60_prior    0.306
VIX_Z60_prior     0.326
VolDiv_prior     -0.059
Name: Forward_RV_30D, dtype: float64


Explained: MOVE_Z = +0.251

*   Explained: MOVE_Z = +0.251
*   我们研究的是：
\[
Corr(MOVE\_Z_t,\ Forward\_RV30_t)=+0.251
\]
*   翻译成人话：
*   在 2004–2022 的数据里，当今天 MOVE 相对于它自己过去60天异常偏高时，接下来30个交易日 SPY 的实际波动率也倾向更高。

*   这里横轴可以想象成：
今天债市有多“不正常”
*   纵轴是：
未来30天SPY实际上有多颠簸

*   The evidence does not support MOVE_Z − VIX_Z as a stable Treasury-to-equity volatility contagion signal.
*  However, component attribution suggests that MOVE and VIX individually contain persistent information about subsequent SPY realized volatility. *  The subtraction-based divergence feature appears to discard rather than enhance that information.

In [34]:
# ==========================================
# POC01-B: Non-overlapping 30D Robustness
# ==========================================

dev_nonoverlap = dev.iloc[::30].copy()
confirm_nonoverlap = confirm.iloc[::30].copy()

features = [
    "MOVE_Z60_prior",
    "VIX_Z60_prior",
    "VolDiv_prior"
]

print("===== SAMPLE SIZE =====")
print("DEV full:", len(dev))
print("DEV non-overlap:", len(dev_nonoverlap))

print("CONFIRM full:", len(confirm))
print("CONFIRM non-overlap:", len(confirm_nonoverlap))


print("\n===== DEVELOPMENT: NON-OVERLAPPING =====")
print(
    dev_nonoverlap[
        features + ["Forward_RV_30D"]
    ]
    .corr()["Forward_RV_30D"]
    .loc[features]
    .round(3)
)


print("\n===== CONFIRMATION: NON-OVERLAPPING =====")
print(
    confirm_nonoverlap[
        features + ["Forward_RV_30D"]
    ]
    .corr()["Forward_RV_30D"]
    .loc[features]
    .round(3)
)

===== SAMPLE SIZE =====
DEV full: 4635
DEV non-overlap: 155
CONFIRM full: 747
CONFIRM non-overlap: 25

===== DEVELOPMENT: NON-OVERLAPPING =====
MOVE_Z60_prior    0.207
VIX_Z60_prior     0.255
VolDiv_prior     -0.043
Name: Forward_RV_30D, dtype: float64

===== CONFIRMATION: NON-OVERLAPPING =====
MOVE_Z60_prior    0.466
VIX_Z60_prior     0.365
VolDiv_prior     -0.037
Name: Forward_RV_30D, dtype: float64


In [35]:
# ==========================================
# POC01-B: 30 Offset Robustness Check
# ==========================================

def offset_correlations(data, horizon=30):

    rows = []

    for offset in range(horizon):

        sample = data.iloc[offset::horizon]

        corr = (
            sample[
                [
                    "MOVE_Z60_prior",
                    "VIX_Z60_prior",
                    "VolDiv_prior",
                    "Forward_RV_30D"
                ]
            ]
            .corr()["Forward_RV_30D"]
        )

        rows.append({
            "Offset": offset,
            "N": len(sample),
            "MOVE_Z": corr["MOVE_Z60_prior"],
            "VIX_Z": corr["VIX_Z60_prior"],
            "VolDiv": corr["VolDiv_prior"]
        })

    return pd.DataFrame(rows)


dev_offsets = offset_correlations(dev)
confirm_offsets = offset_correlations(confirm)


print("===== DEVELOPMENT =====")
print(
    dev_offsets[
        ["MOVE_Z", "VIX_Z", "VolDiv"]
    ].describe().round(3)
)

print("\nPositive correlation share:")
print(
    (dev_offsets[
        ["MOVE_Z", "VIX_Z", "VolDiv"]
    ] > 0).mean().round(3)
)


print("\n===== CONFIRMATION =====")
print(
    confirm_offsets[
        ["MOVE_Z", "VIX_Z", "VolDiv"]
    ].describe().round(3)
)

print("\nPositive correlation share:")
print(
    (confirm_offsets[
        ["MOVE_Z", "VIX_Z", "VolDiv"]
    ] > 0).mean().round(3)
)


===== DEVELOPMENT =====
       MOVE_Z   VIX_Z  VolDiv
count  30.000  30.000  30.000
mean    0.253   0.291  -0.042
std     0.053   0.056   0.043
min     0.172   0.201  -0.124
25%     0.207   0.253  -0.074
50%     0.259   0.286  -0.044
75%     0.293   0.327  -0.009
max     0.348   0.412   0.038

Positive correlation share:
MOVE_Z    1.000
VIX_Z     1.000
VolDiv    0.233
dtype: float64

===== CONFIRMATION =====
       MOVE_Z   VIX_Z  VolDiv
count  30.000  30.000  30.000
mean    0.312   0.328  -0.067
std     0.173   0.197   0.148
min     0.020   0.021  -0.532
25%     0.166   0.143  -0.143
50%     0.266   0.331  -0.044
75%     0.449   0.431   0.013
max     0.612   0.692   0.150

Positive correlation share:
MOVE_Z    1.000
VIX_Z     1.000
VolDiv    0.267
dtype: float64


## POC01 Conclusion — Bond Volatility Contagion

### Original Hypothesis
Unusually high Treasury volatility relative to equity volatility
(MOVE_Z60_prior - VIX_Z60_prior) may signal that equity markets
have not yet incorporated bond-market stress, leading to higher
subsequent SPY risk.

### Result
The hypothesis was not supported.

Across the original 1–5D horizon and the pre-specified 30D
extension, VolDiv did not consistently predict higher subsequent
SPY risk.

The 30D return relationship observed in the development period
(2004–2022) reversed in the confirmation period (2023–2025).

### Component Attribution
MOVE_Z and VIX_Z individually showed positive relationships with
subsequent 30D SPY realized volatility:

Development:
- MOVE_Z: +0.251
- VIX_Z: +0.290
- VolDiv: -0.043

Confirmation:
- MOVE_Z: +0.306
- VIX_Z: +0.326
- VolDiv: -0.059

### Overlap Robustness
Using all 30 possible non-overlapping 30D sampling offsets:

- MOVE_Z was positively correlated with future RV in 30/30
  development samples and 30/30 confirmation samples.
- VIX_Z was positively correlated in 30/30 development and
  30/30 confirmation samples.
- VolDiv was positive in only 23.3% and 26.7%, respectively.

### Interpretation
The subtraction-based divergence feature appears to discard
information rather than enhance it.

The original economic narrative — that unusually high bond
volatility combined with relatively calm equity volatility predicts
a delayed equity-volatility catch-up — is not supported by this POC.

However, the component analysis generates a new exploratory
hypothesis: Treasury volatility itself may contain incremental
information about subsequent equity realized volatility.

This new hypothesis was generated after inspecting POC01 results
and therefore requires a separate research design and should not
be treated as confirmation from POC01.

## Appendix — Historical Extreme VolDiv Observations
The final table lists the largest daily `VolDiv_prior` observations. Consecutive dates should be interpreted as episodes rather than independent events. This appendix is descriptive: it is useful for anomaly/regime interpretation, not evidence that VolDiv predicts subsequent equity risk.


In [36]:
# ==========================================
# POC01 Appendix — Top Historical VolDiv Days
# ==========================================

top_voldiv = (
    df.dropna(subset=["VolDiv_prior"])
      .nlargest(20, "VolDiv_prior")
      [
          [
              "MOVE",
              "VIX",
              "MOVE_Z60_prior",
              "VIX_Z60_prior",
              "VolDiv_prior",
              "SPY"
          ]
      ]
)

print(top_voldiv.round(3))

              MOVE    VIX  MOVE_Z60_prior  VIX_Z60_prior  VolDiv_prior  \
Date                                                                     
2026-09-24  104.58  15.67           5.630         -0.318         5.948   
2009-06-01  190.30  30.04           3.984         -1.458         5.443   
2026-09-23   95.45  15.18           4.535         -0.701         5.236   
2021-02-16   56.78  21.46           4.469         -0.586         5.055   
2016-11-14   89.22  14.48           4.891          0.052         4.839   
2009-06-02  183.90  29.63           3.247         -1.515         4.762   
2010-12-15  125.20  17.94           3.229         -1.299         4.528   
2021-02-19   60.43  22.05           4.164         -0.364         4.528   
2014-10-15  101.30  25.27           9.160          4.732         4.428   
2010-12-17  119.10  16.11           2.212         -2.164         4.376   
2009-06-05  183.40  29.62           2.908         -1.452         4.360   
2010-12-16  122.70  17.39           2.